# Phase 8 - Encoder generalization (GATv2)

**Question:** on the graphs where the framework decides something, does the decision survive a change of aggregator?

- GraphSAGE stays the study's encoder. Nothing here re-fits or re-scores a frozen rule.
- Only the convolution varies: `SAGEConv(mean)` -> `GATv2Conv(4 heads)`. Graph, role graph, features, split, objective and eval are identical.
- Link prediction, K=10, seeds 42-44, seven official variants, **paired** band (`tie_break.py`).


In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from experiments import encoder_transfer as et

STYLE = [{"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
         {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center")]},
         {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center")]}]
show = lambda df: display(df.style.set_table_styles(STYLE).hide(axis="index"))

fourcell = pd.read_csv("results/encoder_transfer_fourcell.csv")
agree = pd.read_csv("results/encoder_transfer_agreement.csv")
perseed = pd.read_csv("results/encoder_transfer_perseed.csv")
PANEL = sorted(agree.dataset)
cross = et.cross(perseed[perseed.dataset.isin(PANEL)], "gatv2_edge", "graphsage_edge")
print(f"{len(PANEL)} graphs compared")

## 1 · The four cells

`GraphSAGE`, `GraphSAGE+aug`, `GATv2`, `GATv2+aug` - same graphs, same splits, same seeds. Only the convolution differs.

`+aug` = the best of the six non-`original` variants for that dataset and encoder, picked by mean AUC.

In [ ]:
fc = fourcell[fourcell.dataset.isin(PANEL)]
w = fc.pivot(index="dataset", columns="encoder", values=["original", "best_aug", "best_aug_signal", "gap", "paired_ratio", "aug_wins"])
w.columns = [f"{b.replace('_edge','')}_{a}" for a, b in w.columns]

# THE 2x2: encoder x graph, averaged over the panel. Means across heterogeneous datasets are a summary, not a metric -
# the per-dataset win counts underneath are what the claims rest on.
cells = {"original": ["graphsage_original", "gatv2_original"], "+aug": ["graphsage_best_aug", "gatv2_best_aug"]}
tbl = pd.DataFrame({k: [w[a].mean(), w[b].mean()] for k, (a, b) in cells.items()}, index=["GraphSAGE", "GATv2"])
tbl["aug - original"] = tbl["+aug"] - tbl["original"]
tbl["aug wins (paired)"] = [f"{int(w.graphsage_aug_wins.sum())}/{len(w)}", f"{int(w.gatv2_aug_wins.sum())}/{len(w)}"]
print(f"MEAN AUC over {len(w)} graphs")
display(tbl.round(4))

# Which of the four cells is the best on each dataset - the same 2x2, counted instead of averaged.
four = w[["graphsage_original", "graphsage_best_aug", "gatv2_original", "gatv2_best_aug"]]
best = four.idxmax(axis=1).value_counts()
cnt = pd.DataFrame([[best.get("graphsage_original", 0), best.get("graphsage_best_aug", 0)],
                    [best.get("gatv2_original", 0), best.get("gatv2_best_aug", 0)]],
                   index=["GraphSAGE", "GATv2"], columns=["original", "+aug"])
print(f"\nBEST OF THE FOUR, counted over {len(w)} graphs")
display(cnt)

**Per dataset.**

In [ ]:
show(w.reset_index()[["dataset", "graphsage_original", "graphsage_best_aug", "graphsage_paired_ratio", "graphsage_aug_wins",
                      "gatv2_original", "gatv2_best_aug", "gatv2_paired_ratio", "gatv2_aug_wins"]])
print("paired_ratio = (aug - original) / paired sem across seeds; aug_wins = ratio > 1.0")

## 2 · The objection, and the answer

> _If GATv2 alone already matches GraphSAGE+aug, the rewiring does nothing an encoder change would not do._

GATv2 on the **original** graph vs GraphSAGE on its **best augmented** graph, paired on the shared split.


In [ ]:
better, tie = cross[cross.paired_ratio > 1], cross[cross.paired_ratio.abs() <= 1]
worse = cross[cross.paired_ratio < -1]
for name, g in [("GATv2 alone BETTER", better), ("indistinguishable", tie), ("GATv2 alone WORSE", worse)]:
    print(f"{name:20s} {len(g):2d}/{len(cross)}  {sorted(g.dataset)}")

# The decomposition that matters: where does augmentation still pay under the stronger encoder?
gv = fourcell[(fourcell.encoder == "gatv2_edge") & fourcell.dataset.isin(PANEL)].set_index("dataset")
for name, ds in [("GATv2 alone >= SAGE+aug", set(better.dataset) | set(tie.dataset)),
                 ("GATv2 alone <  SAGE+aug", set(worse.dataset))]:
    g = gv.loc[sorted(ds & set(gv.index))]
    print(f"\n{name}: {len(g)} graphs -> GATv2+aug beats GATv2 alone on {int(g.aug_wins.sum())}")
    print("   ", sorted(g[g.aug_wins].index))

**Augmentation and a stronger encoder are substitutes, not complements.** Where attention alone recovers the
structure, rewiring adds nothing; where attention alone fails, rewiring is what rescues it.


## 3 · Do the calls survive?

Stage 2 is conditional on stage 1 saying _augment_, so it is only scorable where both arms augment **and** both
resolve a single signal. `stage2_comparable` False = an undecided cell, not a disagreement.


In [ ]:
show(agree)
d = agree[~agree.stage1_agree]
print(f"stage 1 agrees {int(agree.stage1_agree.sum())}/{len(agree)}")
print(f"  disagreements: augment->keep {int(((d.stage1_baseline=='augment') & (d.stage1_encoder=='keep original')).sum())}, "
      f"keep->augment {int(((d.stage1_baseline=='keep original') & (d.stage1_encoder=='augment')).sum())}   {sorted(d.dataset)}")

c = agree[agree.stage2_comparable]
both = c[(c.stage1_baseline == "augment") & (c.stage1_encoder == "augment")]
keep = c[(c.stage1_baseline == "keep original") & (c.stage1_encoder == "keep original")]
print(f"stage 2 agrees {int(agree.stage2_agree.sum())}/{len(c)} decided, of which:")
print(f"  both arms augment (the real test) {int(both.stage2_agree.sum())}/{len(both)}   "
      f"{[(r.dataset, r.stage2_baseline, r.stage2_encoder) for _, r in both.iterrows()]}")
print(f"  both arms keep original           {int(keep.stage2_agree.sum())}/{len(keep)} (trivially 'original')")
print(f"  stage 1 disagrees -> out of scope {len(c) - len(both) - len(keep)}")
print(f"undecided cells: GATv2 {int((~agree.encoder_decided).sum())}/{len(agree)}, GraphSAGE {int((~agree.baseline_decided).sum())}/{len(agree)}")

## 4 · Caveats

- **GATv2 runs on GraphSAGE's hyperparameters** (lr 0.01, 50 epochs, 2 layers, 64 dims). Deliberate - tuning would break the only-the-convolution-changes contract - so a GATv2 loss never means attention is worse, only worse _at these settings_.
- **GATv2's original-graph AUC falls below chance on a few graphs.** Those are among its largest "aug rescues GATv2" gaps, so part of that rescue is GATv2-alone failing, not augmentation succeeding. Cell 2 of §2 prints which.
- 3 seeds, so an undecided cell is an instrument limit, not a finding.
- GIN ran only as a plumbing check (`cora`, 7 variants); its numbers are not interpreted.
- Shared ReLU between layers (GAT papers use ELU); ablations A-D were tuned on `enzymes` under GraphSAGE and are not re-run.
